### P2.3 — Deep Learning Foundations | Python to Gen AI Course
### P2.3.6 — ANN — Artificial Neural Network


---
## 🔁 Quick Recap — Where We Are

In the previous notebook we covered the 4 types of neural networks and the decision table for picking the right one.

Now we build each one — starting with the simplest: **ANN.**

We start with **ANN** — the simplest architecture, used for tabular data.

---
## 🧠 What Is an ANN?

An **Artificial Neural Network (ANN)** is a fully connected network —  
the basic form we have been building from P2.3.3 onwards.

```
Input Layer  →  Hidden Layer(s)  →  Output Layer
  (features)     (fully connected)   (prediction)
```

Every neuron in one layer connects to every neuron in the next.  
Also called: **Fully Connected Network** or **Dense Network** or **MLP (Multi-Layer Perceptron)**.

**When to use ANN:**
```
✅  Tabular data — rows and columns, structured features
✅  Features are independent — order does not matter
✅  Regression or classification task

❌  Not for images — loses spatial information
❌  Not for sequences — has no memory of previous steps
```

Real world: fraud detection, churn prediction, house prices,  
medical diagnosis, loan approval, sales forecasting

<img src="images/ann_architecture.png" width="750"/>


---
## 🆚 ANN vs Linear Regression — Why Not Just Use Regression?

We already used Linear Regression in the ML module for house price prediction.
So why use ANN here?

```
Linear Regression:
  y = w1*x1 + w2*x2 + w3*x3 + b
  → Always a straight line
  → Works well when the relationship is simple and roughly linear

ANN:
  Multiple layers with non-linear activations (ReLU)
  → Can learn curved, complex relationships
  → Works when features interact in non-linear ways
```

**Example of a non-linear interaction:**
```
A 3-bedroom house with 1000 sqft  →  cheaper than expected
A 3-bedroom house with 3000 sqft  →  much more expensive

Bedrooms and size together drive the price — not each feature alone.
Linear Regression misses this. ANN can learn it.
```

```
Rule of thumb:
  Simple, roughly linear problem  →  Linear Regression (faster, easier to explain)
  Complex, non-linear patterns    →  ANN
```

<img src="images/ann_vs_regression.png" width="750"/>

---
## 💻 Let's Build It — ANN for House Price Prediction

## ❓ Wait — Same House Price Problem?

**Yes, SAME PROBLEM. But VERY DIFFERENT PURPOSE.**

| | How Networks Learn | ANN in Practice |
|---|---|---|
| **Goal** | Understand *how* learning works (theory) | Build a *complete workflow* (practice) |
| **Data** | Tiny toy dataset (6 houses, 3 features) | Realistic dataset (18 houses, 4 features) |
| **Focus** | Forward pass → Loss → Backprop → Gradient descent | Split → Train → Evaluate → Inference |
| **Workflow** | Just the raw mechanics | Train/test split, feature scaling, metrics |

We follow the same 4-phase workflow:
```
Phase 1 — Split
Phase 2 — Train
Phase 3 — Evaluate
Phase 4 — Inference
```

In [6]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from tensorflow import keras
from tensorflow.keras import layers
import math

# ── DATA ──────────────────────────────────────────────────────────
# Features: Size (sqft), Bedrooms, Age (years), Floors
X = np.array([
    [1000, 2, 10, 1], [1200, 3,  5, 1], [1500, 3,  8, 2],
    [1800, 4,  3, 2], [2000, 4,  2, 2], [2300, 5,  1, 3],
    [1100, 2,  7, 1], [1400, 3,  6, 2], [1700, 4,  4, 2],
    [1900, 4,  3, 2], [2100, 5,  2, 3], [2500, 5,  1, 3],
    [1050, 2,  9, 1], [1350, 3,  7, 1], [1650, 3,  5, 2],
    [1950, 4,  2, 2], [2200, 5,  1, 3], [2400, 5,  1, 3],
], dtype=float)

# Labels: Price in ₹K
y = np.array([200, 250, 300, 360, 400, 450,
              220, 280, 330, 380, 420, 470,
              210, 265, 315, 390, 440, 460], dtype=float)

print(f"Dataset: {X.shape[0]} houses, {X.shape[1]} features each")

Dataset: 18 houses, 4 features each


In [7]:
# ══════════════════════════════════════════════════════════════════
# PHASE 1 : SPLIT
# ══════════════════════════════════════════════════════════════════
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# ── SCALE THE FEATURES ────────────────────────────────────────────
# Neural networks are sensitive to the scale of inputs
# Size (1000-2500) and Age (1-10) are on very different scales
# StandardScaler brings all features to the same range
# Fit ONLY on training data — never on test data
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"Train samples : {X_train.shape[0]}")
print(f"Test  samples : {X_test.shape[0]}")
print("Features scaled — all inputs now on the same scale")

Train samples : 13
Test  samples : 5
Features scaled — all inputs now on the same scale


In [8]:
# ══════════════════════════════════════════════════════════════════
# PHASE 2 : TRAIN
# ══════════════════════════════════════════════════════════════════

# ── BUILD THE NETWORK ─────────────────────────────────────────────
# Input  : 4 features
# Hidden : 16 neurons (ReLU) → 8 neurons (ReLU)
# Output : 1 neuron — predicting a number (regression)
model = keras.Sequential([
    layers.Dense(16, activation='relu', input_shape=(4,)),
    layers.Dense(8,  activation='relu'),
    layers.Dense(1)   # No activation — regression output
])

# ── COMPILE ───────────────────────────────────────────────────────
# MSE loss — we are predicting a number
# Adam — smart gradient descent optimiser
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss='mse'
)

# ── TRAIN ─────────────────────────────────────────────────────────
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=300,
    batch_size=4,
    verbose=0
)

# Show loss dropping over training
print("=== TRAINING LOSS ===")
for ep in [0, 49, 99, 199, 299]:
    tl = history.history['loss'][ep]
    vl = history.history['val_loss'][ep]
    print(f"  Epoch {ep+1:>3}  Train Loss: {tl:>8.2f}   Test Loss: {vl:>8.2f}")

D:\workspace\pycore\.env_dl_phase\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


=== TRAINING LOSS ===
  Epoch   1  Train Loss: 129954.16   Test Loss: 108311.39
  Epoch  50  Train Loss: 10297.84   Test Loss: 13307.31
  Epoch 100  Train Loss:  1815.93   Test Loss:  2475.58
  Epoch 200  Train Loss:   163.95   Test Loss:   369.81
  Epoch 300  Train Loss:    38.31   Test Loss:   210.14


In [9]:
# ══════════════════════════════════════════════════════════════════
# PHASE 3 : EVALUATE
# ══════════════════════════════════════════════════════════════════
train_preds = model.predict(X_train, verbose=0).flatten()
test_preds  = model.predict(X_test,  verbose=0).flatten()

train_r2   = round(r2_score(y_train, train_preds), 3)
test_r2    = round(r2_score(y_test,  test_preds),  3)
train_rmse = round(math.sqrt(mean_squared_error(y_train, train_preds)), 2)
test_rmse  = round(math.sqrt(mean_squared_error(y_test,  test_preds)),  2)

print("=== PHASE 3 : EVALUATE ===")
print(f"  Train R²   : {train_r2}   (1.0 = perfect)")
print(f"  Test  R²   : {test_r2}")
print(f"  Train RMSE : ₹{train_rmse}K  (average prediction error)")
print(f"  Test  RMSE : ₹{test_rmse}K")

gap = abs(train_r2 - test_r2)
if gap < 0.1:
    print("\n  ✅ Good Fit — train and test scores are close")
elif train_r2 > test_r2 + 0.1:
    print("\n  ⚠️  Overfitting — model memorised training data")
else:
    print("\n  ⚠️  Underfitting — model too simple")

=== PHASE 3 : EVALUATE ===
  Train R²   : 0.995   (1.0 = perfect)
  Test  R²   : 0.972
  Train RMSE : ₹6.01K  (average prediction error)
  Test  RMSE : ₹14.5K

  ✅ Good Fit — train and test scores are close


In [10]:
# ══════════════════════════════════════════════════════════════════
# PHASE 4 : INFERENCE
# Only run after Good Fit confirmed in Phase 3
# ══════════════════════════════════════════════════════════════════

def predict_price(size, beds, age, floors):
    """
    Predict house price from features.
    Input is scaled before prediction — same scaler used in training.
    """
    new_house = np.array([[size, beds, age, floors]])
    new_house_scaled = scaler.transform(new_house)
    price = model.predict(new_house_scaled, verbose=0)[0][0]
    return round(price, 1)

print("=== PHASE 4 : INFERENCE ===")
print(f"House 1600 sqft, 3 bed, 4 yrs, 2 floors  →  ₹{predict_price(1600, 3, 4, 2)}K")
print(f"House 2200 sqft, 4 bed, 2 yrs, 3 floors  →  ₹{predict_price(2200, 4, 2, 3)}K")
print(f"House 1000 sqft, 2 bed, 9 yrs, 1 floor   →  ₹{predict_price(1000, 2, 9, 1)}K")

=== PHASE 4 : INFERENCE ===
House 1600 sqft, 3 bed, 4 yrs, 2 floors  →  ₹318.70001220703125K
House 2200 sqft, 4 bed, 2 yrs, 3 floors  →  ₹422.0K
House 1000 sqft, 2 bed, 9 yrs, 1 floor   →  ₹214.10000610351562K


---
## 📌 One Important Thing — Feature Scaling

You noticed we added `StandardScaler` before training. Here is why:

```
Feature     Range       Without scaling
────────────────────────────────────────
Size        1000–2500   Dominates — large numbers
Bedrooms    2–5         Tiny compared to size
Age         1–10        Tiny compared to size
Floors      1–3         Tiny compared to size
```

Without scaling, the network spends most of its effort on `size`  
because its numbers are 1000x larger — the other features barely register.

**StandardScaler** brings every feature to mean=0, std=1:
```
After scaling:
  Size     →  roughly -1.5 to +1.5
  Bedrooms →  roughly -1.5 to +1.5
  Age      →  roughly -1.5 to +1.5
  Floors   →  roughly -1.5 to +1.5

Now all features have equal influence. Network trains much better.
```

> **Rule:** Always scale features before feeding to a neural network.

<img src="images/feature_scaling.png" width="700"/>

---
## ✅ Summary — What You Learned

| Concept | Key Point |
|---|---|
| ANN | Fully connected network — best for tabular, structured data |
| Also called | Dense Network, MLP, Fully Connected Network |
| When to use | Tabular data — rows and columns, features are independent |
| ANN vs Linear Regression | ANN learns non-linear patterns — regression cannot |
| Feature Scaling | Always scale inputs — StandardScaler before training |
| 4 Phases | Split → Train → Evaluate → Inference — same as ML module |